In [15]:
import pandas as pd

customers = pd.read_csv(
    r"C:\Users\Asus\Desktop\customer_prediction\data\raw\customers_dataset.csv"
)

orders = pd.read_csv(
    r"C:\Users\Asus\Desktop\customer_prediction\data\raw\orders_dataset.csv"
)

order_items = pd.read_csv(
    r"C:\Users\Asus\Desktop\customer_prediction\data\raw\items_dataset.csv"
)

payments = pd.read_csv(
    r"C:\Users\Asus\Desktop\customer_prediction\data\raw\payments_dataset.csv"
)

reviews = pd.read_csv(
    r"C:\Users\Asus\Desktop\customer_prediction\data\raw\reviews_dataset.csv"
)
customers.shape, orders.shape, order_items.shape, payments.shape, reviews.shape

((99441, 5), (99441, 8), (112650, 7), (103886, 5), (99224, 7))

In [16]:
# ##########Customer feature by unique customer ID

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    format="mixed"
)
orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

customer_features = (
    orders.groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique"),
        first_order_date=("order_purchase_timestamp", "min"),
        last_order_date=("order_purchase_timestamp", "max")
    )
    .reset_index()
)

customer_features["customer_lifetime_days"] = (
    customer_features["last_order_date"]
    - customer_features["first_order_date"]
).dt.days

customer_features["order_frequency"] = (
    customer_features["total_orders"]
    / (customer_features["customer_lifetime_days"] + 1)
)

customer_features.head()

,customer_unique_id,total_orders,first_order_date,last_order_date,customer_lifetime_days,order_frequency
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-10-05 10:56:00,2018-10-05 10:56:00,0,1.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-07-05 11:11:00,2018-07-05 11:11:00,0,1.0
2,0000f46a3911fa3c0805444483337064,1,2017-10-03 21:05:00,2017-10-03 21:05:00,0,1.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-12-10 20:29:00,2017-12-10 20:29:00,0,1.0
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:00,2017-11-14 19:45:00,0,1.0


In [18]:
# Customer spending features

orders_payments = orders.merge(
    payments[["order_id", "payment_value"]],
    on="order_id",
    how="left"
)

customer_spending = (
    orders_payments
    .groupby("customer_unique_id")
    .agg(
        total_spending=("payment_value", "sum"),
        average_order_value=("payment_value", "mean")
    )
    .reset_index()
)

customer_features = customer_features.merge(
    customer_spending,
    on="customer_unique_id",
    how="left"
)

customer_features.head()

,customer_unique_id,total_orders,first_order_date,last_order_date,customer_lifetime_days,order_frequency,total_spending_x,average_order_value_x,total_spending_y,average_order_value_y
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-10-05 10:56:00,2018-10-05 10:56:00,0,1.0,141.90,141.90,141.90,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-07-05 11:11:00,2018-07-05 11:11:00,0,1.0,27.19,27.19,27.19,27.19
2,0000f46a3911fa3c0805444483337064,1,2017-10-03 21:05:00,2017-10-03 21:05:00,0,1.0,86.22,86.22,86.22,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-12-10 20:29:00,2017-12-10 20:29:00,0,1.0,43.62,43.62,43.62,43.62
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:00,2017-11-14 19:45:00,0,1.0,196.89,196.89,196.89,196.89


In [19]:
# Customer review features

review_features = (
    reviews.groupby("order_id")["review_score"]
    .mean()
    .reset_index()
)

customer_reviews = (
    orders[["order_id", "customer_unique_id"]]
    .merge(review_features, on="order_id", how="left")
    .groupby("customer_unique_id")["review_score"]
    .mean()
    .reset_index()
)

customer_features = customer_features.merge(
    customer_reviews,
    on="customer_unique_id",
    how="left"
)

customer_features.tail()

,customer_unique_id,total_orders,first_order_date,last_order_date,customer_lifetime_days,order_frequency,total_spending_x,average_order_value_x,total_spending_y,average_order_value_y,review_score
96091,fffcf5a5ff07b0908bd4e2dbc735a684,1,2017-08-06 21:00:00,2017-08-06 21:00:00,0,1.0,2067.42,2067.42,2067.42,2067.42,5.0
96092,fffea47cd6d3cc0a88bd621562a9d061,1,2017-10-12 20:07:00,2017-10-12 20:07:00,0,1.0,84.58,84.58,84.58,84.58,4.0
96093,ffff371b4d645b6ecea244b27531430a,1,2017-07-02 15:49:00,2017-07-02 15:49:00,0,1.0,112.46,112.46,112.46,112.46,5.0
96094,ffff5962728ec6157033ef9805bacc48,1,2018-02-05 15:17:00,2018-02-05 15:17:00,0,1.0,133.69,133.69,133.69,133.69,5.0
96095,ffffd2657e2aad2907e67c3e9daecbeb,1,2017-02-05 20:18:00,2017-02-05 20:18:00,0,1.0,71.56,71.56,71.56,71.56,5.0


In [16]:
customer_features.to_csv(
    r"C:\Users\Asus\Desktop\customer_prediction\data\processed\customer_features.csv",
    index=False
)

In [17]:
customer_features.shape

(96096, 9)

In [20]:
print("Customer features:", customer_features.shape)
print("Customers:", customers.shape)

print(
    "Customers with customer_unique_id:",
    customers["customer_unique_id"].notna().sum()
)

print(
    "Unique customer_unique_id:",
    customers["customer_unique_id"].nunique()
)

Customer features: (96096, 11)
Customers: (99441, 5)
Customers with customer_unique_id: 99441
Unique customer_unique_id: 96096
